# Prepare imports and data

Downloading data using the `mlcroissant` library

In [1]:
import mlcroissant as mlc
import pandas as pd
import re # regex

# downloads the dataset
croissant_dataset = mlc.Dataset('https://www.kaggle.com/datasets/jp797498e/twitter-entity-sentiment-analysis/croissant/download')

  -  [Metadata(Twitter Sentiment Analysis)] Property "http://mlcommons.org/croissant/citeAs" is recommended, but does not exist.


This is the raw data that is produced straight from the dataset

In [2]:
record_sets = croissant_dataset.metadata.record_sets
print(record_sets) # we have two datasets: "twitter_training" and "twitter_validation"

df = pd.DataFrame(croissant_dataset.records(record_set=record_sets[0].uuid))
df.head()

[RecordSet(uuid="twitter_training.csv"), RecordSet(uuid="twitter_validation.csv")]


,twitter_training.csv/2401,twitter_training.csv/Borderlands,twitter_training.csv/Positive,twitter_training.csv/im+getting+on+borderlands+and+i+will+murder+you+all+%2C
0,2401,b'Borderlands',b'Positive',b'I am coming to the borders and I will kill y...
1,2401,b'Borderlands',b'Positive',b'im getting on borderlands and i will kill yo...
2,2401,b'Borderlands',b'Positive',b'im coming on borderlands and i will murder y...
3,2401,b'Borderlands',b'Positive',b'im getting on borderlands 2 and i will murde...
4,2401,b'Borderlands',b'Positive',b'im getting into borderlands and i can murder...


## Initial data analysis

Right now, looks like we have 4 available columns:
- ID??/Class id??? (unsure, not required)
- Company/Genre/Username (again, unsure. the dataset doesn't give a lot of information)
- Sentiment, classified as either `Positive`, `Negative`, `Neutral` or `Irrelevant`. In this dataset, it was recommended that `Irrelevant = Neutral`. Each of the classifications are what they mean, no need to explain. 
- The text content itself. 

To use this model, we only need two pieces of data: 
- Sentiment
- Text content (tokenised)

The others are not required/idk what they do/they probably won't help that much

The text content is not in its correct format - it is currently in a word format. This cannot be analysed, so we must put it through a process called tokenisation. 

Of course, tokenisation is not possible without first cleaning up the data. 

# Quick cleanup of data

Some of the columns are not necessary at all. I will purge them and preppare for preprocessing

In [3]:
df = df.drop(columns=['twitter_training.csv/2401', 'twitter_training.csv/Borderlands']) # remove unnecessary columns
df = df.drop(index=0).reset_index(drop=True) # first one is poorly formatted

# the original data formatting is poorly formatted, so lets format it properly. 
df.columns = ['Sentiment', 'Text Content']

# clean up the b'{}' stuff
df['Sentiment'] = df['Sentiment'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)
df['Text Content'] = df['Text Content'].apply(lambda x: x.decode('utf-8') if isinstance(x, bytes) else x)

# randomise the data so we can check out something else other than the top 5. 
df = df.sample(frac=1).reset_index(drop=True)

# if columns are: ID, Sentiment, Text Content, Text Tokens
df = df[
    ~(
        df["Text Content"].fillna("").str.strip().eq("")
    )
].reset_index(drop=True)

df.head()

,Sentiment,Text Content
0,Positive,Holy fuck please drop the ps5 soon my PS4 is g...
1,Irrelevant,"Ironic too how ""build a wall"" was just a memor..."
2,Neutral,Man why beat the Xbox Canada contest to win an
3,Negative,@FortniteGame what a great game! went into cre...
4,Irrelevant,Aswell had a bang for his buck last night was ...


# Preprocessing

## Imports

In this section, I am using a library called NLTK (Natural Language Toolkit), which contains utilities for being able to convert sentences into tokens.   

In [4]:
import nltk
from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
import re
import contractions

In [5]:
nltk.download('stopwords')
nltk.download('twitter_samples')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package twitter_samples to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/tirbofish/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

## Stopwords

Stopwords are words that carry semantic meaning and can make analysis of the words hard. It includes such articles ("a", "the", ...), conjunctions ("and", "or", ...), prepositions ("in", "on", ...) and other such words. 

By removing stopwords, we are able to remove any tokens/words that do not hold minimal meaning. 

In [6]:
stop_words = set(stopwords.words('english'))

There are some words that can drastically change the meaning if removed. 

For example:

"That is not good" (negative) ----removing stopword---> "That is good" (positive) *[not what we want]*

By excluding the negations from the stopwords, we will be able to keep the sentiment of the text block

In [7]:
negations = {'no', 'not', 'nor', 'neither', 'never', 'none',
            "don't", "won't", "can't", "isn't", "aren't", "wasn't"}
stop_words = stop_words - negations

## Tokenising

By tokenising, we are able to convert a long block of text into individual words. 

In [8]:
tokeniser = TweetTokenizer(strip_handles=True, reduce_len=True)

In [9]:
def clean_text_content(text):
    if pd.isna(text):
        return ""
    text = str(text)

    text = re.sub(r'[^\x00-\x7F]+', '', text)     # remove non-ASCII
    text = re.sub(r'http\S+|www\.\S+', '', text)  # remove URLs
    text = re.sub(r'#\w+', '', text)              # remove hashtags
    text = re.sub(r'&\w+;', '', text)             # remove HTML entities
    text = contractions.fix(text)                 # expand contractions
    text = text.lower().strip()
    return text

df['Mutated Text Content'] = df['Text Content'].apply(clean_text_content)
df.head()

,Sentiment,Text Content,Mutated Text Content
0,Positive,Holy fuck please drop the ps5 soon my PS4 is g...,holy fuck please drop the ps5 soon my ps4 is g...
1,Irrelevant,"Ironic too how ""build a wall"" was just a memor...","ironic too how ""build a wall"" was just a memor..."
2,Neutral,Man why beat the Xbox Canada contest to win an,man why beat the xbox canada contest to win an
3,Negative,@FortniteGame what a great game! went into cre...,@fortnitegame what a great game! went into cre...
4,Irrelevant,Aswell had a bang for his buck last night was ...,aswell had a bang for his buck last night was ...


In [10]:
def tokenise_text_content(text):
    if not text:
        return []
    tokens = tokeniser.tokenize(text)
    tokens = [re.sub(r'[^\w\s]', '', w) for w in tokens]  # remove punctuation
    tokens = [w for w in tokens if w and w not in stop_words]
    return tokens

df["Text Tokens"] = df["Mutated Text Content"].apply(tokenise_text_content)

# remove rows with no usable tokens like empty
df = df[df["Text Tokens"].apply(len) > 0].reset_index(drop=True)

df.head()

,Sentiment,Text Content,Mutated Text Content,Text Tokens
0,Positive,Holy fuck please drop the ps5 soon my PS4 is g...,holy fuck please drop the ps5 soon my ps4 is g...,"[holy, fuck, please, drop, ps5, soon, ps4, goi..."
1,Irrelevant,"Ironic too how ""build a wall"" was just a memor...","ironic too how ""build a wall"" was just a memor...","[ironic, build, wall, memory, trick, keep, add..."
2,Neutral,Man why beat the Xbox Canada contest to win an,man why beat the xbox canada contest to win an,"[man, beat, xbox, canada, contest, win]"
3,Negative,@FortniteGame what a great game! went into cre...,@fortnitegame what a great game! went into cre...,"[great, game, went, creative, entire, game, fr..."
4,Irrelevant,Aswell had a bang for his buck last night was ...,aswell had a bang for his buck last night was ...,"[aswell, bang, buck, last, night, good, laugh,..."


## Classification of sentiment

As previously mentioned, there are ~~4~~ 3 primary types of sentiment classification: 
- Positive
- Neutral/Irrelevant
- Negative

Within the set, we can use multi-class classification as so:
- Positive (2)
- Negative (1) (it only made sense if neutral was 0, not Negative)
- Neutral/Irrelevant (0)

In [11]:
s_map = {
    'Positive': 2,
    'Negative': 1,
    'Neutral': 0,
    'Irrelevant': 0,
}

df['Sentiment'] = df['Sentiment'].map(s_map)
df.head()

,Sentiment,Text Content,Mutated Text Content,Text Tokens
0,2,Holy fuck please drop the ps5 soon my PS4 is g...,holy fuck please drop the ps5 soon my ps4 is g...,"[holy, fuck, please, drop, ps5, soon, ps4, goi..."
1,0,"Ironic too how ""build a wall"" was just a memor...","ironic too how ""build a wall"" was just a memor...","[ironic, build, wall, memory, trick, keep, add..."
2,0,Man why beat the Xbox Canada contest to win an,man why beat the xbox canada contest to win an,"[man, beat, xbox, canada, contest, win]"
3,1,@FortniteGame what a great game! went into cre...,@fortnitegame what a great game! went into cre...,"[great, game, went, creative, entire, game, fr..."
4,0,Aswell had a bang for his buck last night was ...,aswell had a bang for his buck last night was ...,"[aswell, bang, buck, last, night, good, laugh,..."


# Finish

Preprocessing is completed. Let's save into a .csv to be modified in feature_engineering

In [13]:
import os
os.mkdir("csv")
df.to_csv('csv/preprocess.csv', index=False)